# Project 1 — Writing & Debugging Companion 🛠️
### How to build the skill of *fixing things when they break*

You said the goal isn't to write everything from scratch — it's to **understand code well enough to spot and fix what's wrong, efficiently**, and to write clean, reusable functions. This companion trains exactly that, using the house-price project you already understand.

From now on, every project will include these five habits. We practice all of them here:

1. **Read the error** — tracebacks are a map, not noise.
2. **Know the usual suspects** — ~6 errors cause most of your pain. Recognise them on sight.
3. **Break it on purpose** — fix planted bugs (the fastest way to build intuition).
4. **Predict before you run** — guess the output first; a wrong guess reveals a gap in your mental model.
5. **Refactor to production** — turn messy notebook cells into clean, documented functions.

> Run this in Colab or locally, same as the main notebook. Have it open *beside* `house_price_prediction.ipynb`.


## Skill 1 — How to read a Python traceback 🔁

When code crashes, Python prints a **traceback**. Most beginners panic at the wall of text. Here's the trick: **read it bottom-up.**

```
Traceback (most recent call last):
  File "script.py", line 12, in <module>      <- where it started
    result = preprocess(data)
  File "script.py", line 5, in preprocess
    return df["incmoe"]                         <- the line that actually broke
KeyError: 'incmoe'                              <- READ THIS FIRST
```

Three questions, in order:
1. **What is the error type and message?** (the very last line). `KeyError: 'incmoe'` → "you asked for a key/column that doesn't exist."
2. **Which line of *my* code triggered it?** Scan upward to the last file that's *yours* (not a library). Here: `df["incmoe"]`.
3. **Why?** Now you have the *what* and the *where* — usually a typo (`incmoe` → `income`).

> 🧠 *Analogy:* a traceback is a chain of "who called whom." The **bottom** is the scene of the crime; everything above is how you got there. Start at the crime scene.


## Skill 2 — The usual suspects (your quick-reference table) 🔁

These six cause the large majority of errors in ML/pandas/sklearn work. After this project they should feel familiar, not scary.

| Error you see | What it almost always means | First thing to check |
|---|---|---|
| `ModuleNotFoundError: No module named 'x'` | The library isn't installed in this environment | `pip install x` |
| `KeyError: 'colname'` | Column/key doesn't exist (usually a typo or wrong name) | Print `df.columns` and compare |
| `ValueError: could not convert string to float: 'NY'` | A text column reached a model that only takes numbers | Encode categoricals first |
| `ValueError: Input X contains NaN` | Missing values reached the model | Check `df.isnull().sum()`, fill or drop |
| `NotFittedError` | You called `.predict()` / `.transform()` before `.fit()` | Fit the model/scaler first |
| `ValueError: feature names should match...` (shape mismatch) | Train and prediction data have different columns | Apply the *same* preprocessing to both |

The cells below each *deliberately trigger one of these* so you see the real message. Run them, read the error using Skill 1, then look at the fix.


In [ ]:
# Setup for the debugging exercises
import numpy as np, pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = fetch_california_housing(as_frame=True).frame
X = df.drop(columns=["MedHouseVal"]); y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Ready. df has", df.shape[0], "rows.")

## Skill 3 — Break it on purpose 🐛

For each bug: **run the broken cell, predict the error type using the table above, then read the real traceback, then study the fix.** Don't skip running it — *seeing* the real error is what builds the reflex.

### 🐛 Bug 1 — predict before fit
What do you think happens when we ask a model to predict *before* it has learned anything?


In [ ]:
# BROKEN — run me and read the error
model = LinearRegression()
predictions = model.predict(X_test)   # we never called .fit()

**Diagnosis:** `NotFittedError`. The model has no learned weights yet — there's nothing to predict *with*.
🧠 *Like asking someone to take an exam on a subject they never studied.* **Fix:** fit before predict.


In [ ]:
# FIXED
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Worked. First 3 predictions:", np.round(predictions[:3], 2))

### 🐛 Bug 2 — the typo'd column


In [ ]:
# BROKEN — run me and read the error
avg_income = X_train["MedIncome"].mean()   # is that the real column name?

**Diagnosis:** `KeyError: 'MedIncome'`. The actual column is `MedInc`. When you hit a `KeyError`, your fastest move is to print the real names and compare.


In [ ]:
# FIX-IT-YOURSELF: print the columns, find the right name, then compute the mean.
print(list(X_train.columns))
# YOUR CODE HERE: assign the correct mean to avg_income
# avg_income = X_train["..."].mean()
# print(avg_income)

### 🐛 Bug 3 — the silent killer: data leakage 🔁🔁
This one is nastier because **it does NOT throw an error.** The code runs, the score looks amazing, and you ship a model that fails in production. These silent bugs are why "the code ran" never means "the code is correct."


In [ ]:
# SUBTLY WRONG — scaler learns from the WHOLE dataset, including the test set
scaler_bad = StandardScaler()
scaler_bad.fit(X)                          # <-- peeked at the test data
X_train_bad = scaler_bad.transform(X_train)
X_test_bad  = scaler_bad.transform(X_test)
print("Runs fine with no error... which is exactly the danger.")

In [ ]:
# CORRECT — scaler learns from training data ONLY
scaler = StandardScaler()
scaler.fit(X_train)                        # fit on train only
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)       # apply same recipe to test
print("Rule to memorise forever: fit on train, transform on both.")

**How to catch silent bugs:** be suspicious of results that look *too good*. If your test score is near-perfect, your first thought should be "where did I leak?" — not "I'm a genius." 🔁 (You'll apply this exact instinct in every future project.)


## Skill 4 — Predict before you run 🔮

Before running each cell below, **write down your guess** (a sticky note is fine). Checking your guess against reality is how you find the gaps in your understanding.


In [ ]:
# GUESS FIRST: what shape (rows, columns) will X_train be? Then run.
print(X_train.shape)

In [ ]:
# GUESS FIRST: this trains a model and prints its R2 on the TEST set.
# Will it be closer to 0.6 or to 0.99? (Remember: a simple linear model on real, messy data.)
model = LinearRegression().fit(X_train_s, y_train)
print(round(model.score(X_test_s, y_test), 3))

If you guessed ~0.99 — that would have been a red flag for leakage on real data. A score around 0.6 is honest and expected for a plain linear model here. *Calibrating your expectations is itself a debugging skill.*


## Skill 5 — Refactor to production-ready functions 🏭

Notebook code is great for exploring, but you can't *deploy* a pile of loose cells. Production code is organised into **functions**: named, reusable, documented blocks with clear inputs and outputs.

**Why functions matter for deployment:**
- **Reusable** — call `preprocess()` on new data tomorrow without copy-pasting.
- **Testable** — you can check each piece independently when something breaks.
- **Readable** — a function name like `evaluate_model()` documents intent.
- **No leakage by design** — pass `X_train`/`X_test` explicitly so the boundary is obvious.

Below is the entire Project 1 pipeline rewritten as clean functions. Notice the **docstrings** (the `\"\"\"...\"\"\"` text explaining each function) and **type hints** (`X_train: pd.DataFrame`) — both are professional habits that make code self-documenting.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


def load_data() -> tuple[pd.DataFrame, pd.Series]:
    """Load the California housing data and split into features (X) and target (y)."""
    frame = fetch_california_housing(as_frame=True).frame
    X = frame.drop(columns=["MedHouseVal"])
    y = frame["MedHouseVal"]
    return X, y


def make_splits(X: pd.DataFrame, y: pd.Series, test_size: float = 0.2, seed: int = 42):
    """Split into train/test. Keeping this in one place prevents inconsistent splits."""
    return train_test_split(X, y, test_size=test_size, random_state=seed)


def scale_features(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """Standardise features. Fits on TRAIN only, then applies to both -> no leakage."""
    scaler = StandardScaler().fit(X_train)          # learn recipe from train
    return scaler.transform(X_train), scaler.transform(X_test), scaler


def evaluate_model(model, X, y_true) -> dict:
    """Return a tidy dict of metrics so callers don't repeat metric code everywhere."""
    preds = model.predict(X)
    return {"MAE": round(mean_absolute_error(y_true, preds), 3),
            "R2":  round(r2_score(y_true, preds), 3)}


# The whole pipeline now reads like a recipe — each step is one clear line:
X, y = load_data()
X_train, X_test, y_train, y_test = make_splits(X, y)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
print("Test metrics:", evaluate_model(rf, X_test, y_test))

### 🏋️ Your turn — write a function from scratch
This is your first real authoring task. Fill in the body. The docstring tells you exactly what it must do; the function signature tells you what goes in and what comes out. (Solution is in the next cell — try before peeking.)


In [ ]:
def add_rooms_per_person(X: pd.DataFrame) -> pd.DataFrame:
    """Return a COPY of X with one new column 'RoomsPerPerson' = AveRooms / AveOccup.

    Why a copy? Mutating the input in place is a classic source of hard-to-find bugs.
    """
    # YOUR CODE HERE:
    # 1. make a copy of X
    # 2. add the new column
    # 3. return it
    pass

# Test your function (uncomment when ready):
# new_X = add_rooms_per_person(X_train)
# print(new_X[["AveRooms", "AveOccup", "RoomsPerPerson"]].head())

In [ ]:
# SOLUTION — compare after you've attempted it
def add_rooms_per_person(X: pd.DataFrame) -> pd.DataFrame:
    """Return a COPY of X with a new 'RoomsPerPerson' column."""
    X = X.copy()                                  # never mutate the caller's data
    X["RoomsPerPerson"] = X["AveRooms"] / X["AveOccup"]
    return X

new_X = add_rooms_per_person(X_train)
print(new_X[["AveRooms", "AveOccup", "RoomsPerPerson"]].head())

## ✅ What you practised
- **Reading tracebacks** bottom-up: error type → your line → why.
- **The six usual-suspect errors** and the first thing to check for each.
- **Fixing planted bugs**, including the silent, error-free killer: **data leakage** 🔁.
- **Predicting outputs** to surface gaps in your mental model.
- **Refactoring** loose cells into documented, deployable functions — and **writing one yourself**.

### Carry-forward habits (every future project will reinforce these)
1. When something breaks, *read the last line of the traceback first.*
2. "The code ran" ≠ "the code is correct" — be suspicious of results that look too good.
3. Once exploration works, **wrap it in a function** before moving on.

When you've worked through this and the main notebook, say the word and we'll start **Project 2 — Image Classification**, where I'll hand you more of the writing (with scaffolding) and plant a bug or two for you to hunt.
